## c24-c28 - Pipeline Tester
Step-by-step walkthrough of the full optimizer pipeline for a single design candidate. Use to debug individual stages or verify module changes before a full GA run.

# 1. Setup
Resolve paths, load stock and search space. Set `MODEL_PREFIX` to a trained surrogate prefix to enable the GNN stage; leave as `None` to skip it.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (candidate / "config.py").exists():
        REPO_ROOT = candidate
        break

SRC_PATH       = REPO_ROOT / "src"
WORKFLOWS_PATH = REPO_ROOT / "workflows"
for path in (REPO_ROOT, SRC_PATH, WORKFLOWS_PATH):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import config
import c00_headquarter_params as c11_params

MODEL_PREFIX = "ID20260516_182257_LR1e-04_EP200_BS64_PW2.5_ROC0.863"  # e.g. "ID20260516_182257_LR1e-04_EP200_BS64_PW2.5_ROC0.863"

# ── Search space ──────────────────────────────────────────────────────
json_path = config.DATA_IO_PATH / f"search_space_{c11_params.GRID}.json"
if not json_path.exists():
    raise FileNotFoundError(
        f"search_space_{c11_params.GRID}.json not found. "
        "Run c12_15 to generate it."
    )
with open(json_path, "r") as f:
    optimizer_search_space = json.load(f)
print(f"Search space loaded: {len(optimizer_search_space)} variables")

# ── Stock ────────────────────────────────────────────────────────────
stock_type = "A" # new, A or B
stock_path = config.TIMBER_STOCK_PATH / f"complete_timber_{stock_type}.csv"
if not stock_path.exists():
    raise FileNotFoundError(f"Stock CSV not found: {stock_path}. Run c16 to generate it.")

df_input_stock = None
for opts in [
    {"sep": ",", "encoding": "utf-8"},
    {"sep": ";", "encoding": "utf-8"},
    {"sep": ",", "encoding": "latin1"},
    {"sep": ";", "encoding": "latin1"},
]:
    try:
        _df = pd.read_csv(stock_path, **opts)
        if _df.shape[1] > 1:
            df_input_stock = _df
            print(f"Stock loaded: {len(_df)} elements  (sep='{opts['sep']}', encoding='{opts['encoding']}')")
            break
    except Exception:
        pass

if df_input_stock is None:
    raise ValueError("Could not parse stock CSV with tested delimiter/encoding combinations.")

df_input_stock.columns = df_input_stock.columns.str.strip()
display(df_input_stock.head(3))

# 2. Geometry
Generate a random design candidate and compute the truss topology (nodes, members, lengths).

In [ ]:
import matplotlib.pyplot as plt
import config
import c22_stage_geometry as stage_geometry

geometry_out = stage_geometry.run_random_geometry_stage(
    json_path=json_path if "json_path" in globals() else None,
    optimizer_search_space=optimizer_search_space if "optimizer_search_space" in globals() else None,
    sample_id=0,
)

my_random_design = geometry_out["my_random_design"]
vertices_list = geometry_out["vertices_list"]
df_vertices = geometry_out["df_vertices"]
df_edges = geometry_out["df_edges"]
df_geometry_overview = geometry_out["df_geometry_overview"]

print(f"Geometry: {len(df_vertices)} nodes, {len(df_edges)} members")
print(
    f"Length range [m]: {df_geometry_overview['length_m'].min():.3f}"
    f" - {df_geometry_overview['length_m'].max():.3f}"
)
display(df_geometry_overview[["edge_id", "V1", "V2", "length_m"]].head(5))

df_vertices.to_csv(config.DATA_IO_PATH / "df_vertices.csv", index=False)
df_edges.to_csv(config.DATA_IO_PATH / "df_edges.csv", index=False)

fig, ax = stage_geometry.plot_geometry_preview(
    df_vertices=df_vertices,
    df_edges=df_edges,
    figsize=(8, 7),
)
plt.show()

## 3–7. Pipeline stages
Run the full pipeline on the geometry from section 2: feasibility filter → cost matrix → MILP → GNN → fitness.

In [ ]:
import importlib
import json
import numpy as np
import pandas as pd
import config
from workflows import c24_stage_feasibility as stage_feas
from workflows import c25_stage_cost_matrix as stage_cost
from workflows import c26_stage_MILP        as stage_milp

importlib.reload(stage_feas)
importlib.reload(stage_cost)
importlib.reload(stage_milp)

verts = df_vertices[df_vertices['sample_id'] == 0].copy()
verts['v_idx'] = verts['vertex_index'].str.replace('v', '').astype(int)
verts = verts.sort_values('v_idx').reset_index(drop=True)
node_positions = verts[['x', 'y', 'z']].values

support_nodes = verts[verts['attribute'] == 'support']['v_idx'].tolist()
load_nodes    = verts[verts['attribute'] == 'load']['v_idx'].tolist()

df_slots, feasibility_mask, member_forces, stats = stage_feas.build_cost_filter(
    node_positions=node_positions,
    edges_df=df_edges,
    stock_df=df_input_stock,
    support_nodes=support_nodes,
    load_nodes=load_nodes,
)

n_feasible = feasibility_mask.sum()
n_total    = feasibility_mask.size
print(f"Feasibility: {n_feasible:,} / {n_total:,} pairs feasible  ({100*n_feasible/n_total:.1f}%)")
print(f"Forces — tension: {(member_forces > 1.0).sum()}  compression: {(member_forces < -1.0).sum()}  "
      f"near-zero: {(np.abs(member_forces) <= 1.0).sum()}")

In [ ]:
# STEP 1: COST MATRIX STAGE
cost_matrix, stock_prepared, logs = stage_cost.build_cost_matrix(
    df_slots=df_slots,
    df_input_stock=df_input_stock,
    feasibility_mask=feasibility_mask,
)

cost_matrix_df = pd.DataFrame(cost_matrix)
cost_matrix_df.to_csv(config.EXPORT_PATH / "c26_cost_matrix.csv")

# STEP 2: MILP STAGE
print("Starting MILP optimizer...")
milp_out = stage_milp.run_milp_stage(
    cost_matrix=cost_matrix,
    enriched_stock=stock_prepared,
    df_slots=df_slots,
    reclaimed_marker="RS",
    new_marker="NS",
    new_stock_max_uses=100,
    solver_msg=False,
    raise_on_infeasible_slots=False,
    )

status = milp_out.get("status")
df_results = milp_out.get("df_results", pd.DataFrame())
total_cost = milp_out.get("total_cost", float("inf"))
milp_summary = milp_out.get("summary", {})

df_results.to_csv(config.EXPORT_PATH / "c27_milp_results.csv", index=False)

print(
    f"MILP setup: {milp_summary.get('reclaimed_items', 0)} reclaimed + "
    f"{milp_summary.get('new_items', 0)} new stock items for {milp_summary.get('slots', len(df_slots))} slots"
)
print(f"MILP status: {status}")
print(f"Total assignment cost: {total_cost:.2f}")
if status != "Optimal":
    print("MILP did not solve to an optimal assignment for this geometry and stock set.")
if len(df_results) > 0:
    display(df_results.head(10))

# 4b. Cost breakdown - RS vs NS analysis
Rebuilds the cost matrix with full logs enabled to decompose each LCA component for reclaimed (RS) vs new (NS) stock. Shows why RS elements cost more (or less) than NS on a per-slot basis.

In [ ]:
import importlib
import numpy as np
import pandas as pd
import config
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from workflows import c24_stage_feasibility as stage_feas
from workflows import c25_stage_cost_matrix as stage_cost
importlib.reload(stage_feas)
importlib.reload(stage_cost)

import c00_headquarter_params as _p

COMPONENTS = ["E_embodied", "E_recovered", "E_transport", "E_prep", "E_saw", "E_waste", "E_scarcity"]
LABELS     = ["Embodied\n(A1-A3)", "Recovery\n(C1)", "Transport\n(A4)", "Prep\n(A5-p)", "Saw\n(A5-s)", "Waste\n(C2+C3-C4)", "Scarcity"]

STOCK_FILES = {
    "Stock A": "complete_timber_A.csv",
    "Stock B": "complete_timber_B.csv",
}
# Colors: NS shared baseline + RS per stock
C_NS   = config.PLOT_COLORS.get("NS", "#61788C")
C_RS_A = config.PLOT_COLORS.get("RS", "#F2994B")
C_RS_B = config.PLOT_COLORS.get("danger", "#D9653B")

# ── Load both stocks ──────────────────────────────────────────────────────────
dfs = {}
for label, fname in STOCK_FILES.items():
    path = config.TIMBER_STOCK_PATH / fname
    for opts in [{"sep": ";", "encoding": "utf-8"}, {"sep": ",", "encoding": "utf-8"},
                 {"sep": ";", "encoding": "latin1"}, {"sep": ",", "encoding": "latin1"}]:
        try:
            _df = pd.read_csv(path, **opts)
            if _df.shape[1] > 1:
                _df.columns = _df.columns.str.strip()
                dfs[label] = _df
                ns_n = int((_df["State"] == 0).sum())
                rs_n = int((_df["State"] == 1).sum())
                print(f"{label}: {len(_df)} elements  NS={ns_n}  RS={rs_n}")
                break
        except Exception:
            pass

# ── Strategy: NS elements are identical in both CSVs.
#    Run the full pipeline for each stock separately, but treat NS as a
#    shared baseline — only one NS reference is needed.
#    Comparison shows: NS (shared) | RS Stock A | RS Stock B
# ─────────────────────────────────────────────────────────────────────────────
rs_analysis = {}
ns_by_branch = None   # filled from first run

for label, df_stock in dfs.items():
    print(f"\n{label}:")
    slots, feas_mask, _, _ = stage_feas.build_cost_filter(
        node_positions = node_positions,
        edges_df       = df_edges,
        stock_df       = df_stock,
        support_nodes  = support_nodes,
        load_nodes     = load_nodes,
    )
    _, sp, df_logs = stage_cost.build_cost_matrix(
        df_slots         = slots,
        df_input_stock   = df_stock,
        feasibility_mask = feas_mask,
        build_logs       = True,
    )
    feasible  = df_logs[df_logs["Feasible"]].copy()
    by_branch = feasible.groupby("Branch")[COMPONENTS + ["Total cost"]].mean()

    # Capture shared NS reference from first run only
    if ns_by_branch is None and "new" in by_branch.index:
        ns_by_branch = by_branch.loc[["new"]]
        n_ns_ref = int((feasible["Branch"] == "new").sum())
        ns_t_ref = float(by_branch.loc["new", "Total cost"])
        print(f"  NS reference set: {n_ns_ref:,} pairs  mean={ns_t_ref:.5f} kg CO2e/pair")

    # RS analysis for this stock
    rs_feasible = feasible[feasible["Branch"] == "reclaimed"].copy()
    n_rs = len(rs_feasible)
    rs_t = float(by_branch.loc["reclaimed", "Total cost"]) if "reclaimed" in by_branch.index else float("nan")

    rs_min   = rs_feasible.groupby("Slot_ID")["Total cost"].min().rename("RS_min")
    ns_min   = feasible[feasible["Branch"] == "new"].groupby("Slot_ID")["Total cost"].min().rename("NS_min")
    rs_waste = rs_feasible.groupby("Slot_ID")["V_waste_m3"].mean().rename("RS_waste_m3")
    slot_cmp = pd.concat([ns_min, rs_min, rs_waste], axis=1).dropna(subset=["NS_min", "RS_min"]).copy()
    slot_cmp["RS_premium"] = slot_cmp["RS_min"] - slot_cmp["NS_min"]
    slot_cmp["RS_cheaper"] = slot_cmp["RS_premium"] < 0

    n_cheaper = int(slot_cmp["RS_cheaper"].sum())
    rs_analysis[label] = {
        "by_branch": by_branch,
        "slot_cmp":  slot_cmp,
        "n_rs": n_rs,
        "n_cheaper": n_cheaper,
        "rs_t": rs_t,
    }
    print(f"  RS pairs: {n_rs:,}  mean RS cost: {rs_t:.5f} kg CO2e/pair  "
          f"(vs NS {ns_t_ref:.5f}  →  {'+' if rs_t>ns_t_ref else ''}{100*(rs_t-ns_t_ref)/ns_t_ref:.1f}%)")
    print(f"  RS cheaper than NS: {n_cheaper}/{len(slot_cmp)} slots ({100*n_cheaper/len(slot_cmp):.1f}%)")
    print(f"  Mean RS premium: {slot_cmp['RS_premium'].mean():+.4f} kg CO2e")

print(f"\nLCA factors: A1-A3={_p.IMPACT_FACTOR_A1_A3}  C1={_p.IMPACT_FACTOR_RECOVERED_C1}  "
      f"A5-prep={_p.ENERGY_PREP_A5}  A5-saw={_p.ENERGY_SAW_A5}")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.52, wspace=0.38)

labels_list = list(rs_analysis.keys())
RS_COLORS   = {"Stock A": C_RS_A, "Stock B": C_RS_B}

# Panel A — LCA component breakdown: NS (shared) vs RS A vs RS B
ax0 = fig.add_subplot(gs[0, :2])
active     = [c for c in COMPONENTS if
              (ns_by_branch[c].abs().max() > 1e-7) or
              any(rs_analysis[lbl]["by_branch"].get(c, pd.Series([0])).abs().max() > 1e-7
                  for lbl in labels_list)]
active_lbl = [LABELS[COMPONENTS.index(c)] for c in active]
x = np.arange(len(active))
bar_w = 0.28

# NS bar (shared reference)
ns_vals = [float(ns_by_branch.loc["new", c]) if "new" in ns_by_branch.index else 0.0 for c in active]
bars_ns = ax0.bar(x - bar_w, ns_vals, bar_w, label=f"NS (shared, n={n_ns_ref:,})", color=C_NS, alpha=0.85)
for bar in bars_ns:
    h = bar.get_height()
    if h > 1e-5:
        ax0.text(bar.get_x()+bar.get_width()/2, h+0.0001, f"{h:.4f}", ha="center", va="bottom", fontsize=6.5)

# RS bars per stock
for i, lbl in enumerate(labels_list):
    by_b = rs_analysis[lbl]["by_branch"]
    rs_vals = [float(by_b.loc["reclaimed", c]) if "reclaimed" in by_b.index else 0.0 for c in active]
    offset = 0.0 + i * bar_w
    bars = ax0.bar(x + offset, rs_vals, bar_w,
                   label=f"{lbl} RS (n={rs_analysis[lbl]['n_rs']:,})",
                   color=RS_COLORS[lbl], alpha=0.85)
    for bar in bars:
        h = bar.get_height()
        if h > 1e-5:
            ax0.text(bar.get_x()+bar.get_width()/2, h+0.0001, f"{h:.4f}", ha="center", va="bottom", fontsize=6.5)

ax0.set_xticks(x); ax0.set_xticklabels(active_lbl, fontsize=9)
ax0.set_ylabel("Mean LCA cost (kg CO2e / pair)")
ax0.set_title("A — LCA component breakdown: NS baseline vs RS from Stock A vs Stock B")
ax0.legend(fontsize=9); ax0.grid(axis="y", alpha=0.3)

# Panel B — Total mean cost: NS | RS A | RS B
ax1 = fig.add_subplot(gs[0, 2])
bar_labels_b = ["NS\n(shared)", "RS\nStock A", "RS\nStock B"]
bar_vals_b   = [ns_t_ref, rs_analysis["Stock A"]["rs_t"], rs_analysis["Stock B"]["rs_t"]]
bar_colors_b = [C_NS, C_RS_A, C_RS_B]
bars_b = ax1.bar(bar_labels_b, bar_vals_b, color=bar_colors_b, alpha=0.85, width=0.55)
ax1.set_ylabel("Mean total LCA cost (kg CO2e / pair)")
ax1.set_title("B — Mean total cost\nNS vs RS per donor building")
for bar in bars_b:
    h = bar.get_height()
    ax1.text(bar.get_x()+bar.get_width()/2, h+0.0001, f"{h:.4f}",
             ha="center", va="bottom", fontsize=8.5, fontweight="bold")
ax1.axhline(ns_t_ref, color=C_NS, lw=1.2, linestyle="--", alpha=0.6)
ax1.grid(axis="y", alpha=0.3)

# Panel C — RS premium per slot (both stocks overlaid)
ax2 = fig.add_subplot(gs[1, :2])
for lbl in labels_list:
    sorted_prem = rs_analysis[lbl]["slot_cmp"].sort_values("RS_premium")["RS_premium"].values
    x_norm = np.linspace(0, 1, len(sorted_prem))
    n_ch  = rs_analysis[lbl]["n_cheaper"]
    n_tot = len(sorted_prem)
    ax2.plot(x_norm, sorted_prem, color=RS_COLORS[lbl], lw=2, alpha=0.85,
             label=f"{lbl}  [{n_ch}/{n_tot} slots RS cheaper]")
    ax2.fill_between(x_norm, sorted_prem, 0, where=(sorted_prem < 0),
                     color=RS_COLORS[lbl], alpha=0.10)
ax2.axhline(0, color="k", lw=1.0)
ax2.set_xlabel("Slots (sorted by RS premium, normalised to [0, 1])")
ax2.set_ylabel("min(RS cost) − min(NS cost)  [kg CO2e]")
ax2.set_title("C — RS cost premium per slot: Stock A vs Stock B\n(below 0 = RS cheaper than the cheapest NS option for that slot)")
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

# Panel D — Summary table
ax3 = fig.add_subplot(gs[1, 2])
ax3.axis("off")
row_labels = ["RS pairs", "Mean RS cost", "RS vs NS", "RS cheaper slots", "Mean RS premium"]
col_data = []
for lbl in labels_list:
    a = rs_analysis[lbl]
    pct = 100 * (a["rs_t"] - ns_t_ref) / ns_t_ref if ns_t_ref > 1e-9 else float("nan")
    col_data.append([
        f"{a['n_rs']:,}",
        f"{a['rs_t']:.4f}",
        f"{pct:+.1f}% vs NS",
        f"{a['n_cheaper']}/{len(a['slot_cmp'])} ({100*a['n_cheaper']/len(a['slot_cmp']):.0f}%)",
        f"{a['slot_cmp']['RS_premium'].mean():+.4f}",
    ])
tbl = ax3.table(
    cellText=[[col_data[0][i], col_data[1][i]] for i in range(len(row_labels))],
    rowLabels=row_labels,
    colLabels=labels_list,
    cellLoc="center", loc="center",
    colWidths=[0.25, 0.25]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 1.65)
ax3.set_title("D — Summary comparison")

fig.suptitle(
    f"LCA Cost Comparison: Donor Building A vs B — Same Design, {len(df_edges)} Members\n"
    f"NS baseline is shared (same 421 NS catalog elements in both stocks)",
    fontsize=14, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.show()


In [ ]:
import c27_stage_GNN as stage_gnn
from c21_surrogate_io import load_surrogate_bundle

model_bundle = None
milp_status = milp_out.get("status")
milp_assignment = milp_out.get("milp_assignment")

if milp_status != "Optimal" or milp_assignment is None:
    print(f"Skipping GNN stage because MILP status is {milp_status!r}.")
elif MODEL_PREFIX:
    print(f"Loading surrogate model bundle with prefix: {MODEL_PREFIX}")
    model_bundle = load_surrogate_bundle(prefix_sm=MODEL_PREFIX)
else:
    model_bundle = globals().get("model_bundle")
    if model_bundle is None:
        print("No MODEL_PREFIX set and no model_bundle loaded; skipping GNN stage.")

if model_bundle is None:
    gnn_out = {
        "feasibility_score": 1.0,
        "structural_penalty": 0.0,
        "unsafe_member_ids": [],
    }
else:
    gnn_out = stage_gnn.run_gnn_stage(
        node_positions=node_positions,
        milp_assignment=milp_assignment,
        df_input_stock=df_input_stock,
        model_bundle=model_bundle,
    )

feasibility_score = float(gnn_out.get("feasibility_score", 1.0))
structural_penalty = float(gnn_out.get("structural_penalty", 0.0))
unsafe_member_ids = gnn_out.get("unsafe_member_ids", [])

print(f"\nGNN Feasibility Results:")
print(f"  Feasibility score:  {feasibility_score:.4f}  (1.0 = all members predicted safe)")
print(f"  Unsafe members:     {len(unsafe_member_ids)} / {stage_gnn.NUM_EDGES_PHYSICAL}")
print(f"  Structural penalty: {structural_penalty:.4f}  (w_structural=0.3)")
if unsafe_member_ids:
    print(f"  Unsafe member IDs:  {unsafe_member_ids[:20]}{'...' if len(unsafe_member_ids) > 20 else ''}")


In [ ]:
from workflows.c28_stage_fitness_score import run_fitness_stage
from workflows.c28_stage_normalization_bounds import run_normalization_bounds_stage

# ── Normalisation bounds ──────────────────────────────────────────────────────
bounds_out = run_normalization_bounds_stage(
    cost_matrix        = cost_matrix,
    df_logs            = globals().get('df_logs', logs),
    enriched_stock     = stock_prepared,
    df_slots           = df_slots,
    reclaimed_marker   = "RS",
    new_marker         = "NS",
    new_stock_max_uses = 100,
    solver_msg         = False,
    print_summary      = True,
)
normalization_constants = bounds_out["normalization_constants"]
print(normalization_constants)

# ── Fitness weights ───────────────────────────────────────────────────────────
# Single-shot evaluation — omega_4 fixed at w_start. The GA schedules it dynamically.
w_start = 0.2
w_end   = 0.8

fitness_weights = {
    "omega_1": 1.0,
    "omega_2": 1.0,
    "omega_3": 1.0,
    "omega_4": w_start,
}

# ── Fitness stage ─────────────────────────────────────────────────────────────
if df_results.empty:
    print("Skipping fitness stage — MILP returned no assignments.")
    fitness_result = {"objective": None, "is_feasible": False, "fitness": float("inf")}
    fitness_out = {"fitness_result": fitness_result, "normalization_constants": normalization_constants}
else:
    fitness_out = run_fitness_stage(
        df_results               = df_results,
        enriched_stock           = stock_prepared,
        df_slots                 = df_slots,
        total_cost               = total_cost,
        weight_config            = fitness_weights,
        normalization_constants  = normalization_constants,
        structural_infeasibility = 1.0 - feasibility_score,
        derive_normalization_constants = False,
        run_sanity_checks        = True,
        print_breakdown          = True,
    )
    fitness_result          = fitness_out["fitness_result"]
    normalization_constants = fitness_out["normalization_constants"]

fitness_export = dict(fitness_out)

print(f"\nFitness summary:")
print(f"  objective: {fitness_result.get('objective', 'n/a')}")
print(f"  feasible:  {fitness_result.get('is_feasible', 'n/a')}")
print(f"  fitness:   {fitness_result.get('fitness', 'n/a')}")

# ── Export ────────────────────────────────────────────────────────────────────
fitness_json_path = config.EXPORT_PATH / "c28_fitness_result.json"
fitness_csv_path  = config.EXPORT_PATH / "c28_fitness_result.csv"
fitness_json_path.parent.mkdir(parents=True, exist_ok=True)

def _to_builtin(value):
    return value.item() if hasattr(value, "item") else value

can_export = (not df_results.empty) and all(
    v == v and v not in (float("inf"), float("-inf"))
    for v in normalization_constants.values()
)

if can_export:
    fitness_export.pop("weights", None)
    fitness_export["weights"] = {k: float(v) for k, v in fitness_weights.items()}
    fitness_export["normalization_constants"] = {k: float(v) for k, v in normalization_constants.items()}
    with open(fitness_json_path, "w", encoding="utf-8") as f:
        json.dump(fitness_export, f, indent=2)
    fitness_row = {
        **{k: _to_builtin(v) for k, v in fitness_result.items()},
        **fitness_weights,
        **{k: float(v) for k, v in normalization_constants.items()},
    }
    pd.DataFrame([fitness_row]).to_csv(fitness_csv_path, index=False)
    print(f"\nExported: {fitness_json_path.name}, {fitness_csv_path.name}")
else:
    print("Skipping fitness export — pipeline infeasible or normalisation bounds unavailable.")


# 8. Export
Write vertices and edges (with MILP assignments) to disk for Grasshopper reconstruction. Skipped if MILP returned no assignment.


In [ ]:
EXPORT_PREFIX = "c29_optimum"

if any(name not in globals() for name in ["df_vertices", "df_edges", "df_results"]) or df_results.empty:
    print("Skipping structural export — no MILP assignment available.")
else:
    df_edges_export = pd.merge(
        df_edges,
        df_results[["edge_id", "assigned_timber", "CO2_Penalty"]],
        on="edge_id", how="left",
    )
    df_edges_export["assigned_timber"] = df_edges_export["assigned_timber"].fillna("UNASSIGNED")
    df_edges_export["CO2_Penalty"]     = df_edges_export["CO2_Penalty"].fillna(0)

    df_vertices.to_csv(config.EXPORT_PATH / f"{EXPORT_PREFIX}_vertices.csv", index=False)
    df_edges_export.to_csv(config.EXPORT_PATH / f"{EXPORT_PREFIX}_edges.csv", index=False)

    n_matched = int((df_edges_export["assigned_timber"] != "UNASSIGNED").sum())
    print(f"Exported: {len(df_vertices)} vertices, {len(df_edges_export)} edges ({n_matched} matched)")
